In [2]:
import numpy as np

# Example gradient functions: assume f_i(x) = 0.5 * (a_i^T x - b_i)^2
def make_grad_f_list(A, b):
    return [lambda x, a=a_i, bi=bi: a * (a @ x - bi) for a_i, bi in zip(A, b)]

# Example proximal operator for L2 regularization: h(x) = (lambda_/2) * ||x||^2
def prox_l2(x, eta, lambda_=1.0):
    return x / (1 + eta * lambda_)

# VRPDA² algorithm implementation
def vrpda2(
    grad_f_list,          # List of gradient functions [∇f1, ∇f2, ..., ∇fn]
    prox_h,               # Proximal operator for h(x)
    x0,                   # Initial primal variable (numpy array)
    eta,                  # Step size
    T,                    # Number of iterations
    lambda_=1.0,          # Regularization parameter for proximal
    update_table=True     # Whether to update stored gradients
):
    n = len(grad_f_list)
    d = len(x0)
    x = x0.copy()
    y = np.zeros_like(x)
    x_hist = [x.copy()]

    # Initialize gradient table
    grad_table = np.array([grad_f(x) for grad_f in grad_f_list])
    avg_grad = np.mean(grad_table, axis=0)

    for t in range(T):
        i = np.random.randint(n)
        grad_i_current = grad_f_list[i](x)
        grad_i_previous = grad_table[i]

        v = grad_i_current - grad_i_previous + avg_grad
        y += eta * v
        x = prox_h(x0 - y, eta, lambda_)

        if update_table:
            grad_table[i] = grad_i_current
            avg_grad = np.mean(grad_table, axis=0)

        x_hist.append(x.copy())

    return x, x_hist

# Example usage
if __name__ == "__main__":
    np.random.seed(0)
    n, d = 10, 5
    A = np.random.randn(n, d)
    b = np.random.randn(n)
    grad_f_list = make_grad_f_list(A, b)

    x0 = np.zeros(d)
    eta = 0.1
    T = 50

    x_final, x_hist = vrpda2(grad_f_list, prox_l2, x0, eta, T)
    print("Final solution x:", x_final)

Final solution x: [-0.82343745  0.23470581 -0.24327505 -0.64679426  0.20016367]
